# 📸 gotohp — Google Photos Quota Reclaim (Colab Web UI)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ntrungz/gotohp/blob/main/colab/gotohp_reclaim.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-ntrungz%2Fgotohp-181717?logo=github)](https://github.com/ntrungz/gotohp)

Công cụ chạy trực tiếp trên **Google Colab** với giao diện Form & Dashboard trực quan giúp:
1. **Quét toàn bộ thư viện Google Photos** để tìm các ảnh/video đang chiếm dung lượng (`QuotaChargedBytes > 0`).
2. **Tải bản gốc về thư mục backup** (Colab hoặc Google Drive).
3. **Đưa bản cũ vào Thùng rác (Trash)** — có tùy chọn xóa vĩnh viễn khỏi Trash để tránh Google Photos trùng SHA-1.
4. **Upload lại không tốn dung lượng** (`Pixel XL` chất lượng gốc hoặc `Pixel 2` Storage Saver) và khôi phục `Favorite`, `Archived`, `Caption`.

---

> ### ⚠️ Lưu ý & Ghi nhận bản quyền (Attribution & Disclaimers)
> - **Nguồn gốc Fork**: Repository [`ntrungz/gotohp`](https://github.com/ntrungz/gotohp) là bản fork từ dự án gốc [`xob0t/gotohp`](https://github.com/xob0t/gotohp) (MIT License) và tham khảo giao thức mobile API từ [`xob0t/gpmc`](https://github.com/xob0t/gpmc). Không liên kết hay được bảo trợ bởi Google LLC.
> - **An toàn dữ liệu**: Luôn chạy **Bước 3 (Dry-Run)** trước, sau đó thử **Bước 4** với `LIMIT = 2` và `DELETE_PERMANENTLY = False` trên vài ảnh mẫu.
> - **Trash & Trùng SHA-1**: Mặc định chỉ đưa vào Trash (an toàn, có tự động khôi phục từ Trash nếu upload lỗi). Nếu Google Photos báo trùng SHA-1 với bản đang nằm trong Trash, hãy bật `DELETE_PERMANENTLY = True` và gõ `YES` vào ô xác nhận.
> - **Bảo mật**: Không chia sẻ notebook có chứa token/cookie của bạn. Hãy dùng **Bước 6** để xóa config khỏi VM Colab sau khi dùng xong.

In [ ]:
#@title 1️⃣ Cài đặt môi trường & Build gotohp-cli { display-mode: "form" }
#@markdown Chạy cell này đầu tiên để cài Go 1.27, tải mã nguồn `gotohp` và biên dịch `gotohp-cli`.
REPO_URL = "https://github.com/ntrungz/gotohp.git" #@param {type:"string"}
BRANCH = "main" #@param {type:"string"}
MOUNT_GOOGLE_DRIVE = False #@param {type:"boolean"}
REBUILD_FORCE = False #@param {type:"boolean"}

import html
import json
import os
import shutil
import subprocess
import time
from pathlib import Path
from IPython.display import HTML, display

WORK_DIR = Path("/content/gotohp")
GOTOHP_BIN = Path("/content/gotohp-cli")
GO_TARBALL = "go1.27.0.linux-amd64.tar.gz"
GO_BIN_DIR = "/usr/local/go/bin"

if GO_BIN_DIR not in os.environ.get("PATH", ""):
    os.environ["PATH"] = f"{GO_BIN_DIR}:{os.environ.get('PATH', '')}"


def format_bytes(num_bytes: int) -> str:
    if not num_bytes or num_bytes <= 0:
        return "0 B"
    val = float(num_bytes)
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if val < 1024 or unit == "TB":
            return f"{val:.1f} {unit}"
        val /= 1024
    return f"{val:.1f} TB"


def render_card(title: str, body_html: str, accent: str = "#3b82f6") -> HTML:
    return HTML(
        f"""
        <div style="font-family: system-ui, -apple-system, sans-serif; border: 1px solid #334155;
                    border-left: 4px solid {accent}; border-radius: 10px; padding: 14px 16px;
                    background: #0f172a; color: #e2e8f0; margin: 8px 0;">
          <div style="font-weight: 700; font-size: 15px; margin-bottom: 6px; color: #f8fafc;">{title}</div>
          <div style="font-size: 13px; line-height: 1.55;">{body_html}</div>
        </div>
        """
    )


def render_kpi_grid(cards: list[tuple[str, str, str, str]]) -> str:
    items = []
    for label, value, sub, color in cards:
        items.append(
            f"""
            <div style="flex: 1 1 160px; background: #1e293b; border: 1px solid #334155;
                        border-top: 3px solid {color}; border-radius: 10px; padding: 12px 14px;">
              <div style="font-size: 11px; text-transform: uppercase; letter-spacing: 0.04em; color: #94a3b8;">{html.escape(label)}</div>
              <div style="font-size: 22px; font-weight: 700; color: #f8fafc; margin: 4px 0;">{html.escape(value)}</div>
              <div style="font-size: 11px; color: #cbd5e1;">{html.escape(sub)}</div>
            </div>
            """
        )
    return f'<div style="display: flex; flex-wrap: wrap; gap: 10px; margin: 10px 0;">{"".join(items)}</div>'


status_handle = display(render_card("⚙️ Đang khởi tạo...", "Kiểm tra môi trường Go & mã nguồn..."), display_id=True)
steps_log = []


def log_step(msg: str, accent: str = "#3b82f6"):
    steps_log.append(msg)
    status_handle.update(
        render_card(
            "⚙️ Thiết lập gotohp trên Google Colab",
            "<br>".join(steps_log),
            accent=accent,
        )
    )


try:
    # 1. Mount Google Drive (optional)
    DEFAULT_BACKUP_DIR = "/content/gotohp-reclaim"
    if MOUNT_GOOGLE_DRIVE:
        log_step("📁 Đang kết nối Google Drive...")
        from google.colab import drive  # type: ignore

        drive.mount("/content/drive", force_remount=False)
        DEFAULT_BACKUP_DIR = "/content/drive/MyDrive/gotohp-reclaim"
        os.makedirs(DEFAULT_BACKUP_DIR, exist_ok=True)
        log_step(f"✅ Đã gắn Google Drive — thư mục backup mặc định: <code>{html.escape(DEFAULT_BACKUP_DIR)}</code>")
    else:
        os.makedirs(DEFAULT_BACKUP_DIR, exist_ok=True)
        log_step(f"📁 Thư mục backup mặc định trên VM Colab: <code>{html.escape(DEFAULT_BACKUP_DIR)}</code>")

    # 2. Install Go 1.27 if missing or older
    need_go = True
    if shutil.which("go"):
        ver_out = subprocess.check_output(["go", "version"], text=True).strip()
        if "go1.27" in ver_out or "go1.28" in ver_out:
            need_go = False
            log_step(f"✅ Đã có sẵn Go: <code>{html.escape(ver_out)}</code>")

    if need_go:
        log_step("⬇️ Đang tải & cài đặt Go 1.27.0 (~75 MB)...")
        subprocess.run(
            ["wget", "-q", f"https://go.dev/dl/{GO_TARBALL}", "-O", f"/tmp/{GO_TARBALL}"],
            check=True,
        )
        subprocess.run(["rm", "-rf", "/usr/local/go"], check=True)
        subprocess.run(["tar", "-C", "/usr/local", "-xzf", f"/tmp/{GO_TARBALL}"], check=True)
        subprocess.run(["rm", "-f", f"/tmp/{GO_TARBALL}"], check=True)
        ver_out = subprocess.check_output(["go", "version"], text=True).strip()
        log_step(f"✅ Đã cài Go: <code>{html.escape(ver_out)}</code>")

    # 3. Clone or update repository
    if not WORK_DIR.exists():
        log_step(f"📦 Đang clone <code>{html.escape(REPO_URL)}</code> (nhánh <code>{html.escape(BRANCH)}</code>)...")
        subprocess.run(
            ["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(WORK_DIR)],
            check=True,
        )
    else:
        log_step("🔄 Đang cập nhật mã nguồn mới nhất...")
        subprocess.run(["git", "-C", str(WORK_DIR), "fetch", "origin", BRANCH, "--depth", "1"], check=True)
        subprocess.run(["git", "-C", str(WORK_DIR), "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)

    # 4. Build gotohp-cli
    if REBUILD_FORCE or not GOTOHP_BIN.exists():
        log_step("🔨 Đang biên dịch <code>gotohp-cli</code> (chỉ mất ~15-30 giây)...")
        t0 = time.time()
        subprocess.run(
            ["go", "build", "-tags", "cli", "-o", str(GOTOHP_BIN), "."],
            cwd=str(WORK_DIR),
            check=True,
        )
        log_step(f"✅ Biên dịch thành công sau {time.time() - t0:.1f}s → <code>{GOTOHP_BIN}</code>", accent="#10b981")
    else:
        log_step(f"✅ Đã có sẵn binary <code>{GOTOHP_BIN}</code> (bật REBUILD_FORCE nếu muốn build lại)", accent="#10b981")

    help_check = subprocess.run([str(GOTOHP_BIN), "version"], capture_output=True, text=True)
    log_step(f"🚀 Sẵn sàng! Phiên bản: <code>{html.escape(help_check.stdout.strip() or 'gotohp-cli')}</code>", accent="#10b981")
except Exception as exc:
    log_step(f"❌ Lỗi thiết lập: <code>{html.escape(str(exc))}</code>", accent="#ef4444")
    raise

In [ ]:
#@title 2️⃣ Kết nối tài khoản Google Photos (Ẩn token an toàn) { display-mode: "form" }
#@markdown Chọn phương thức đăng nhập. Token được truyền qua `stdin`, **không bao giờ in ra màn hình hay lưu trong output**.
AUTH_SOURCE = "Nhập trực tiếp (ẩn ký tự - khuyên dùng)" #@param ["Nhập trực tiếp (ẩn ký tự - khuyên dùng)", "Đọc từ Colab Secrets", "Chỉ xem danh sách tài khoản hiện có"]
COLAB_SECRET_NAME = "GOTOHP_CRED" #@param {type:"string"}
PROXY = "" #@param {type:"string"}
SET_ACTIVE_ACCOUNT = "" #@param {type:"string"}

import getpass
import html
import subprocess
from IPython.display import HTML, display

if not GOTOHP_BIN.exists():
    raise RuntimeError("Vui lòng chạy Bước 1️⃣ để cài đặt gotohp-cli trước!")

display(
    render_card(
        "🔑 Hướng dẫn nhanh lấy Credential (1 trong 2 cách)",
        """
        <b>Cách 1 — Embedded Setup (Nhanh, không cần Android):</b><br>
        1. Mở <a href="https://accounts.google.com/EmbeddedSetup" target="_blank" style="color:#60a5fa;">accounts.google.com/EmbeddedSetup</a> và đăng nhập, bấm <b>I agree</b> (trang xoay mãi là bình thường).<br>
        2. Mở DevTools (<code>F12</code>) → tab <b>Application</b> → <b>Cookies</b> → <code>accounts.google.com</code> → copy giá trị cookie <code>oauth_token</code>.<br><br>
        <b>Cách 2 — Google Photos ReVanced:</b><br>
        Dán trực tiếp chuỗi <code>androidId=...&amp;Email=...&amp;Token=...</code> lấy từ logcat GmsCore.
        """,
        accent="#8b5cf6",
    )
)

secret_value = ""
if AUTH_SOURCE == "Nhập trực tiếp (ẩn ký tự - khuyên dùng)":
    secret_value = getpass.getpass("Dán oauth_token hoặc chuỗi androidId=... rồi nhấn Enter: ").strip()
elif AUTH_SOURCE == "Đọc từ Colab Secrets":
    try:
        from google.colab import userdata  # type: ignore

        secret_value = (userdata.get(COLAB_SECRET_NAME) or "").strip()
        if not secret_value:
            raise ValueError(f"Secret '{COLAB_SECRET_NAME}' đang trống.")
    except Exception as e:
        display(
            render_card(
                "❌ Không đọc được Colab Secret",
                f"Hãy mở biểu tượng 🔑 <b>Secrets</b> ở thanh bên trái Colab, tạo key <code>{html.escape(COLAB_SECRET_NAME)}</code> và bật quyền truy cập cho notebook.<br>Chi tiết: <code>{html.escape(str(e))}</code>",
                accent="#ef4444",
            )
        )
        raise

if secret_value:
    cmd = [str(GOTOHP_BIN), "creds", "add", "-"]
    if PROXY.strip():
        cmd += ["--proxy", PROXY.strip()]
    proc = subprocess.run(cmd, input=secret_value + "\n", capture_output=True, text=True)
    del secret_value
    if proc.returncode != 0:
        err_msg = (proc.stderr or proc.stdout).strip()
        display(render_card("❌ Kết nối tài khoản thất bại", f"<code>{html.escape(err_msg)}</code>", accent="#ef4444"))
        raise RuntimeError(err_msg)
    display(render_card("✅ Đã kết nối tài khoản thành công", f"<code>{html.escape(proc.stdout.strip())}</code>", accent="#10b981"))

if SET_ACTIVE_ACCOUNT.strip():
    proc = subprocess.run([str(GOTOHP_BIN), "creds", "set", SET_ACTIVE_ACCOUNT.strip()], capture_output=True, text=True)
    if proc.returncode != 0:
        display(render_card("⚠️ Không đổi được tài khoản active", f"<code>{html.escape((proc.stderr or proc.stdout).strip())}</code>", accent="#f59e0b"))
    else:
        display(render_card("✅ Đã chuyển tài khoản active", f"<code>{html.escape(proc.stdout.strip())}</code>", accent="#10b981"))

# Show connected accounts visually
ls_proc = subprocess.run([str(GOTOHP_BIN), "creds", "ls"], capture_output=True, text=True)
accounts_html = []
for raw_line in ls_proc.stdout.splitlines():
    line = raw_line.strip()
    if line.startswith("* "):
        email = line[2:].strip()
        accounts_html.append(
            f'<div style="padding:6px 10px; margin:4px 0; border-radius:6px; background:#064e3b; color:#a7f3d0;">'
            f'🟢 <b>{html.escape(email)}</b> <span style="font-size:11px; opacity:0.85;">(Đang chọn - Active)</span></div>'
        )
    elif "@" in line and not line.startswith("Use "):
        accounts_html.append(
            f'<div style="padding:6px 10px; margin:4px 0; border-radius:6px; background:#1e293b; color:#cbd5e1;">'
            f'⚪ {html.escape(line)}</div>'
        )

if accounts_html:
    display(render_card("👤 Tài khoản Google Photos đã lưu", "".join(accounts_html), accent="#10b981"))
else:
    display(render_card("ℹ️ Chưa có tài khoản nào", "Chọn phương thức nhập ở form phía trên rồi chạy lại cell này.", accent="#f59e0b"))

In [ ]:
#@title 3️⃣ Quét thư viện & Xem báo cáo dung lượng (Dry-Run — chỉ đọc) { display-mode: "form" }
#@markdown Quét toàn bộ thư viện Google Photos để liệt kê các ảnh/video đang chiếm dung lượng. **Không tải, không xóa, không sửa bất cứ gì.**
ACCOUNT = "" #@param {type:"string"}
SCAN_PROXY = "" #@param {type:"string"}
PREVIEW_TOP_N = 50 #@param {type:"slider", min:10, max:200, step:10}
EXPORT_SCAN_JSON = True #@param {type:"boolean"}

import html
import json
import subprocess
import threading
import ipywidgets as widgets
from IPython.display import HTML, display

if not GOTOHP_BIN.exists():
    raise RuntimeError("Vui lòng chạy Bước 1️⃣ trước!")

cmd = [str(GOTOHP_BIN), "reclaim", "--dry-run", "--json-progress"]
if ACCOUNT.strip():
    cmd += ["--account", ACCOUNT.strip()]
if SCAN_PROXY.strip():
    cmd += ["--proxy", SCAN_PROXY.strip()]

live_box = display(
    render_card("🔍 Đang quét thư viện Google Photos...", "Đang gửi yêu cầu trang đầu tiên..."),
    display_id=True,
)

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, bufsize=1)
stdout_chunks = []


def _read_stdout():
    assert proc.stdout is not None
    stdout_chunks.append(proc.stdout.read())


t_out = threading.Thread(target=_read_stdout, daemon=True)
t_out.start()

stderr_errors = []
assert proc.stderr is not None
for line in proc.stderr:
    line = line.strip()
    if not line:
        continue
    try:
        evt = json.loads(line)
        if evt.get("event") == "scan_progress":
            s = evt.get("data", {})
            live_box.update(
                render_card(
                    "🔍 Đang quét thư viện Google Photos (Live)...",
                    render_kpi_grid(
                        [
                            ("Trang đã quét", str(s.get("pagesScanned", 0)), "Đang phân trang...", "#38bdf8"),
                            ("Tổng file kiểm tra", f"{s.get('totalScanned', 0):,}", "Ảnh & Video", "#818cf8"),
                            ("File ăn dung lượng", f"{s.get('eligibleCount', 0):,}", "Đủ điều kiện thu hồi", "#f59e0b"),
                            ("Có thể thu hồi", format_bytes(s.get("totalQuotaBytes", 0)), "Ước tính giải phóng", "#10b981"),
                        ]
                    ),
                    accent="#38bdf8",
                )
            )
    except json.JSONDecodeError:
        stderr_errors.append(line)

proc.wait()
t_out.join()

raw_stdout = "".join(stdout_chunks).strip()
if proc.returncode != 0 and not raw_stdout:
    err_text = "\n".join(stderr_errors) or "Scan failed without output"
    live_box.update(render_card("❌ Quét thư viện thất bại", f"<pre style='white-space:pre-wrap;'>{html.escape(err_text)}</pre>", accent="#ef4444"))
    raise RuntimeError(err_text)

LAST_SCAN_DATA = json.loads(raw_stdout)
scan_obj = LAST_SCAN_DATA.get("scan") or {}
if scan_obj.get("errorMessage"):
    live_box.update(render_card("❌ Lỗi khi quét thư viện", f"<code>{html.escape(scan_obj['errorMessage'])}</code>", accent="#ef4444"))
    raise RuntimeError(scan_obj["errorMessage"])

stats = scan_obj.get("stats") or {}
items = scan_obj.get("items") or []

if EXPORT_SCAN_JSON:
    report_path = "/content/gotohp_scan_report.json"
    with open(report_path, "w", encoding="utf-8") as f:
        json.dump(LAST_SCAN_DATA, f, ensure_ascii=False, indent=2)

# Breakdown stats
orig_count = sum(1 for it in items if it.get("isOriginalQuality"))
saver_count = len(items) - orig_count
video_count = sum(1 for it in items if it.get("type") == 2)
photo_count = len(items) - video_count
skipped_total = (
    stats.get("skippedTrashed", 0)
    + stats.get("skippedLocked", 0)
    + stats.get("skippedNotOwned", 0)
    + stats.get("skippedNoDedupId", 0)
)

summary_html = render_kpi_grid(
    [
        ("Tổng file đã quét", f"{stats.get('totalScanned', 0):,}", f"{stats.get('pagesScanned', 0)} trang", "#818cf8"),
        ("File ăn dung lượng", f"{len(items):,}", f"Ảnh: {photo_count:,} · Video: {video_count:,}", "#f59e0b"),
        ("Dung lượng thu hồi", format_bytes(stats.get("totalQuotaBytes", 0)), f"Gốc: {orig_count:,} · Saver: {saver_count:,}", "#10b981"),
        (
            "Đã bỏ qua an toàn",
            f"{skipped_total:,}",
            f"Trash: {stats.get('skippedTrashed', 0)} · Locked: {stats.get('skippedLocked', 0)} · Shared: {stats.get('skippedNotOwned', 0)}",
            "#64748b",
        ),
    ]
)

if EXPORT_SCAN_JSON:
    summary_html += '<div style="font-size:12px; color:#94a3b8; margin-top:4px;">💾 Đã lưu báo cáo đầy đủ tại <code>/content/gotohp_scan_report.json</code></div>'

live_box.update(render_card("✅ Quét thư viện hoàn tất", summary_html, accent="#10b981"))

# Interactive filterable preview table using ipywidgets
if items:
    search_input = widgets.Text(
        value="",
        placeholder="Tìm theo tên file (VD: .mp4, IMG_2024...)",
        description="🔎 Lọc:",
        layout=widgets.Layout(width="340px"),
    )
    quality_filter = widgets.Dropdown(
        options=[("Tất cả chất lượng", "all"), ("Chỉ Original Quality", "orig"), ("Chỉ Storage Saver", "saver")],
        value="all",
        description="Chất lượng:",
        layout=widgets.Layout(width="240px"),
    )
    table_out = widgets.HTML()

    def _render_table(_=None):
        q = search_input.value.strip().lower()
        qf = quality_filter.value
        filtered = []
        for it in items:
            if q and q not in (it.get("fileName") or "").lower():
                continue
            if qf == "orig" and not it.get("isOriginalQuality"):
                continue
            if qf == "saver" and it.get("isOriginalQuality"):
                continue
            filtered.append(it)

        shown = filtered[:PREVIEW_TOP_N]
        rows = []
        for idx, it in enumerate(shown, 1):
            badges = []
            if it.get("isOriginalQuality"):
                badges.append('<span style="background:#1d4ed8;color:#dbeafe;padding:1px 6px;border-radius:4px;font-size:11px;">Original</span>')
            else:
                badges.append('<span style="background:#334155;color:#cbd5e1;padding:1px 6px;border-radius:4px;font-size:11px;">Saver</span>')
            if it.get("type") == 2:
                badges.append('<span style="background:#7c2d12;color:#fed7aa;padding:1px 6px;border-radius:4px;font-size:11px;">Video</span>')
            if it.get("isFavorite"):
                badges.append('<span style="background:#854d0e;color:#fef08a;padding:1px 6px;border-radius:4px;font-size:11px;">⭐ Favorite</span>')
            if it.get("isArchived"):
                badges.append('<span style="background:#312e81;color:#e0e7ff;padding:1px 6px;border-radius:4px;font-size:11px;">📦 Archived</span>')
            if it.get("isEdited"):
                badges.append('<span style="background:#581c87;color:#f3e8ff;padding:1px 6px;border-radius:4px;font-size:11px;">✏️ Edited</span>')
            if it.get("isMicroVideo"):
                badges.append('<span style="background:#155e75;color:#cffafe;padding:1px 6px;border-radius:4px;font-size:11px;">🎬 Motion</span>')

            rows.append(
                f"""
                <tr style="border-bottom:1px solid #1e293b;">
                  <td style="padding:7px 8px; color:#94a3b8;">{idx}</td>
                  <td style="padding:7px 8px; color:#f8fafc; font-family:monospace; max-width:340px; overflow:hidden; text-overflow:ellipsis; white-space:nowrap;">{html.escape(it.get('fileName') or '')}</td>
                  <td style="padding:7px 8px; color:#fbbf24; font-weight:600; text-align:right; white-space:nowrap;">{format_bytes(it.get('quotaChargedBytes', 0))}</td>
                  <td style="padding:7px 8px;">{" ".join(badges)}</td>
                </tr>
                """
            )

        table_out.value = f"""
        <div style="font-family:system-ui,-apple-system,sans-serif; background:#0f172a; border:1px solid #334155; border-radius:10px; overflow:hidden; margin-top:6px;">
          <div style="padding:8px 12px; background:#1e293b; color:#cbd5e1; font-size:12px;">
            Hiển thị <b>{len(shown)}</b> / <b>{len(filtered):,}</b> file khớp bộ lọc (sắp xếp từ nặng nhất đến nhẹ nhất)
          </div>
          <div style="max-height:360px; overflow-y:auto;">
            <table style="width:100%; border-collapse:collapse; font-size:12.5px;">
              <thead>
                <tr style="background:#020617; color:#94a3b8; text-align:left;">
                  <th style="padding:8px;">#</th>
                  <th style="padding:8px;">Tên file</th>
                  <th style="padding:8px; text-align:right;">Ăn dung lượng</th>
                  <th style="padding:8px;">Thuộc tính</th>
                </tr>
              </thead>
              <tbody>{"".join(rows) or '<tr><td colspan="4" style="padding:16px;text-align:center;color:#94a3b8;">Không có file nào khớp bộ lọc</td></tr>'}</tbody>
            </table>
          </div>
        </div>
        """

    search_input.observe(_render_table, names="value")
    quality_filter.observe(_render_table, names="value")
    _render_table()
    display(widgets.VBox([widgets.HBox([search_input, quality_filter]), table_out]))
else:
    display(render_card("🎉 Tuyệt vời!", "Tài khoản của bạn hiện không có file nào đang chiếm dung lượng Google Photos.", accent="#10b981"))

In [ ]:
#@title 4️⃣ Chạy Thu hồi Dung lượng (Download → Trash → Re-upload 0 Quota) { display-mode: "form" }
#@markdown ⚠️ **Khuyên dùng lần đầu**: Để `LIMIT = 2`, `DELETE_PERMANENTLY = False`, `CLEANUP_LOCAL = False` để test an toàn.
#@markdown Nếu Google Photos báo trùng SHA-1 với bản trong Trash, bật `DELETE_PERMANENTLY = True` và gõ `YES` ở ô xác nhận cuối cùng.
RECLAIM_ACCOUNT = "" #@param {type:"string"}
DOWNLOAD_DIR = "" #@param {type:"string"}
LIMIT = 2 #@param {type:"integer"}
THREADS = 1 #@param {type:"slider", min:1, max:5, step:1}
DELETE_PERMANENTLY = False #@param {type:"boolean"}
CLEANUP_LOCAL = False #@param {type:"boolean"}
STORAGE_SAVER = False #@param {type:"boolean"}
RESTORE_METADATA = True #@param {type:"boolean"}
RECLAIM_PROXY = "" #@param {type:"string"}
CONFIRM_PERMANENT_DELETE = "" #@param {type:"string"}

import html
import json
import subprocess
import threading
import ipywidgets as widgets
from IPython.display import HTML, display

if not GOTOHP_BIN.exists():
    raise RuntimeError("Vui lòng chạy Bước 1️⃣ trước!")

if DELETE_PERMANENTLY and CONFIRM_PERMANENT_DELETE.strip().upper() != "YES":
    display(
        render_card(
            "🛑 Chốt an toàn: Cần xác nhận xóa vĩnh viễn",
            "Bạn đang bật <code>DELETE_PERMANENTLY = True</code> (xóa vĩnh viễn bản cũ khỏi Trash trước khi up lại).<br>"
            "Để tránh bấm nhầm, hãy gõ <b>YES</b> vào ô <code>CONFIRM_PERMANENT_DELETE</code> rồi chạy lại cell này.",
            accent="#ef4444",
        )
    )
    raise SystemExit("Confirmation required for permanent delete")

effective_download_dir = DOWNLOAD_DIR.strip() or globals().get("DEFAULT_BACKUP_DIR", "/content/gotohp-reclaim")

cmd = [
    str(GOTOHP_BIN),
    "reclaim",
    "--json-progress",
    "--download-dir",
    effective_download_dir,
    "--threads",
    str(max(1, int(THREADS))),
]
if RECLAIM_ACCOUNT.strip():
    cmd += ["--account", RECLAIM_ACCOUNT.strip()]
if int(LIMIT) > 0:
    cmd += ["--limit", str(int(LIMIT))]
if DELETE_PERMANENTLY:
    cmd += ["--delete-permanently"]
if CLEANUP_LOCAL:
    cmd += ["--cleanup-local"]
if STORAGE_SAVER:
    cmd += ["--saver"]
if not RESTORE_METADATA:
    cmd += ["--no-restore-metadata"]
if RECLAIM_PROXY.strip():
    cmd += ["--proxy", RECLAIM_PROXY.strip()]

# Config preview card
mode_badge = "🔥 Trash + Xóa vĩnh viễn khỏi Trash" if DELETE_PERMANENTLY else "🗑️ Chỉ đưa vào Trash (có tự động rollback nếu lỗi)"
qual_badge = "📱 Storage Saver (Pixel 2 - 0 quota)" if STORAGE_SAVER else "💎 Chất lượng gốc Original (Pixel XL - 0 quota)"
local_badge = "🧹 Tự xóa file tạm sau khi up xong" if CLEANUP_LOCAL else f"💾 Giữ backup tại <code>{html.escape(effective_download_dir)}</code>"

display(
    render_card(
        "📋 Cấu hình đợt Reclaim hiện tại",
        f"• <b>Chế độ xóa bản cũ</b>: {mode_badge}<br>"
        f"• <b>Chất lượng up lại</b>: {qual_badge}<br>"
        f"• <b>Backup trên máy</b>: {local_badge}<br>"
        f"• <b>Giới hạn</b>: {'Tất cả' if int(LIMIT) == 0 else f'{int(LIMIT)} file'} · <b>Luồng</b>: {int(THREADS)} · <b>Khôi phục metadata</b>: {'Có' if RESTORE_METADATA else 'Không'}",
        accent="#6366f1",
    )
)

# Live widgets dashboard
prog_bar = widgets.IntProgress(
    value=0,
    min=0,
    max=1,
    description="Tiến độ:",
    bar_style="info",
    layout=widgets.Layout(width="100%"),
)
live_status_html = widgets.HTML(value="<div style='color:#94a3b8;font-size:13px;'>🔍 Đang quét thư viện...</div>")
live_log_html = widgets.HTML(value="")
display(widgets.VBox([prog_bar, live_status_html, live_log_html]))

STAGE_LABELS = {
    "downloading": "⬇️ Đang tải bản gốc",
    "trashing": "🗑️ Đang đưa vào Trash",
    "deleting": "🔥 Đang xóa khỏi Trash",
    "uploading": "⬆️ Đang up lại (0 quota)",
    "restoring_metadata": "🏷️ Khôi phục metadata",
    "restoring_trash": "↩️ Khôi phục từ Trash",
    "cleaning_up": "🧹 Dọn file tạm",
    "completed": "✅ Hoàn tất",
    "error": "❌ Lỗi",
}

state = {
    "total": 0,
    "total_bytes": 0,
    "processed": 0,
    "succeeded": 0,
    "failed": 0,
    "reclaimed_bytes": 0,
    "workers": {},
    "recent": [],
}


def _refresh_live_ui(phase_text: str = ""):
    worker_lines = []
    for wid in sorted(state["workers"].keys()):
        w = state["workers"][wid]
        st = STAGE_LABELS.get(w.get("stage", ""), w.get("stage", ""))
        b_done = w.get("bytesDone", 0)
        b_tot = w.get("bytesTotal", 0)
        pct = f" ({min(100, int(b_done * 100 / b_tot))}%)" if b_tot > 0 else ""
        worker_lines.append(
            f"<div style='padding:4px 8px;margin:3px 0;background:#1e293b;border-radius:6px;font-size:12px;'>"
            f"<b>[Worker #{wid}]</b> {st}{pct} — <code>{html.escape(w.get('fileName', ''))}</code> "
            f"<span style='color:#94a3b8;'>({html.escape(w.get('message', ''))})</span></div>"
        )

    header = phase_text or (
        f"Đã xử lý: <b>{state['processed']}/{state['total']}</b> · "
        f"✅ Thành công: <b style='color:#34d399;'>{state['succeeded']}</b> · "
        f"❌ Lỗi: <b style='color:#f87171;'>{state['failed']}</b> · "
        f"💾 Đã thu hồi: <b style='color:#fbbf24;'>{format_bytes(state['reclaimed_bytes'])}</b> / {format_bytes(state['total_bytes'])}"
    )
    live_status_html.value = (
        f"<div style='font-family:system-ui;background:#0f172a;border:1px solid #334155;border-radius:8px;padding:10px 12px;color:#e2e8f0;margin-top:4px;'>"
        f"<div style='font-size:13px;margin-bottom:6px;'>{header}</div>"
        f"{''.join(worker_lines)}</div>"
    )

    if state["recent"]:
        recent_rows = []
        for r in reversed(state["recent"][-15:]):
            icon = "❌" if r.get("isError") else "✅"
            detail = html.escape(r.get("errorMessage") or f"Thu hồi {format_bytes(r.get('quotaChargedBytes', 0))} · New key: {r.get('newMediaKey', '')[:16]}...")
            if r.get("restoredFromTrash"):
                detail += " (↩️ Đã tự khôi phục từ Trash)"
            recent_rows.append(
                f"<div style='padding:4px 8px;border-bottom:1px solid #1e293b;font-size:12px;'>"
                f"{icon} <b>{html.escape(r.get('fileName', ''))}</b> — <span style='color:#cbd5e1;'>{detail}</span></div>"
            )
        live_log_html.value = (
            f"<div style='font-family:system-ui;background:#020617;border:1px solid #334155;border-radius:8px;margin-top:6px;max-height:220px;overflow-y:auto;color:#e2e8f0;'>"
            f"<div style='padding:6px 8px;background:#1e293b;font-size:11px;color:#94a3b8;'>Nhật ký xử lý gần nhất (15 file mới nhất)</div>"
            f"{''.join(recent_rows)}</div>"
        )


proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, bufsize=1)
stdout_chunks = []


def _read_stdout():
    assert proc.stdout is not None
    stdout_chunks.append(proc.stdout.read())


t_out = threading.Thread(target=_read_stdout, daemon=True)
t_out.start()

stderr_raw = []
assert proc.stderr is not None
for line in proc.stderr:
    line = line.strip()
    if not line:
        continue
    try:
        evt = json.loads(line)
    except json.JSONDecodeError:
        stderr_raw.append(line)
        continue

    etype = evt.get("event")
    data = evt.get("data") or {}
    if etype == "scan_progress":
        _refresh_live_ui(
            f"🔍 Đang quét thư viện... Trang: <b>{data.get('pagesScanned', 0)}</b> · "
            f"Đã kiểm tra: <b>{data.get('totalScanned', 0):,}</b> · "
            f"Tìm thấy: <b>{data.get('eligibleCount', 0):,}</b> ({format_bytes(data.get('totalQuotaBytes', 0))})"
        )
    elif etype == "reclaim_start":
        state["total"] = data.get("total", 0)
        state["total_bytes"] = data.get("totalQuotaBytes", 0)
        prog_bar.max = max(1, state["total"])
        prog_bar.value = 0
        _refresh_live_ui()
    elif etype == "item_progress":
        state["workers"][data.get("workerId", 0)] = data
        _refresh_live_ui()
    elif etype == "item_result":
        state["processed"] += 1
        if data.get("isError"):
            state["failed"] += 1
        else:
            state["succeeded"] += 1
            state["reclaimed_bytes"] += data.get("quotaChargedBytes", 0)
        state["recent"].append(data)
        prog_bar.value = min(prog_bar.max, state["processed"])
        _refresh_live_ui()

proc.wait()
t_out.join()

raw_stdout = "".join(stdout_chunks).strip()
if not raw_stdout:
    prog_bar.bar_style = "danger"
    err = "\n".join(stderr_raw) or "Process exited without output"
    display(render_card("❌ Lỗi chạy Reclaim", f"<pre style='white-space:pre-wrap;'>{html.escape(err)}</pre>", accent="#ef4444"))
    raise RuntimeError(err)

LAST_RECLAIM_OUTPUT = json.loads(raw_stdout)
with open("/content/gotohp_reclaim_result.json", "w", encoding="utf-8") as f:
    json.dump(LAST_RECLAIM_OUTPUT, f, ensure_ascii=False, indent=2)

summary = LAST_RECLAIM_OUTPUT.get("summary") or {}
has_err = bool(summary.get("errorMessage")) or summary.get("failed", 0) > 0
prog_bar.value = prog_bar.max
prog_bar.bar_style = "warning" if has_err else "success"

final_html = render_kpi_grid(
    [
        ("Tổng mục xử lý", str(summary.get("total", 0)), "Trong đợt này", "#818cf8"),
        ("Thành công", str(summary.get("succeeded", 0)), "Đã up lại 0 quota", "#10b981"),
        ("Thất bại", str(summary.get("failed", 0)), "Giữ nguyên backup local", "#ef4444" if summary.get("failed", 0) else "#64748b"),
        ("Dung lượng thu hồi", format_bytes(summary.get("reclaimedBytes", 0)), f"Backup: {summary.get('downloadDir', effective_download_dir)}", "#f59e0b"),
    ]
)
if summary.get("errorMessage"):
    final_html += f"<div style='color:#fca5a5;margin-top:6px;'>⚠️ {html.escape(summary['errorMessage'])}</div>"
final_html += "<div style='font-size:12px;color:#94a3b8;margin-top:4px;'>📄 Chi tiết đầy đủ đã lưu tại <code>/content/gotohp_reclaim_result.json</code></div>"

display(
    render_card(
        "⚠️ Hoàn tất có lỗi một phần" if has_err else "🎉 Thu hồi dung lượng hoàn tất!",
        final_html,
        accent="#f59e0b" if has_err else "#10b981",
    )
)

In [ ]:
#@title 5️⃣ (Tùy chọn) Upload thư mục từ Colab / Google Drive lên Google Photos (0 Quota) { display-mode: "form" }
#@markdown Dùng cell này nếu bạn muốn upload trực tiếp ảnh/video từ Google Drive hoặc thư mục trên Colab lên Google Photos không tốn dung lượng.
UPLOAD_SOURCE_PATH = "/content/drive/MyDrive/Photos" #@param {type:"string"}
UPLOAD_ACCOUNT = "" #@param {type:"string"}
RECURSIVE = True #@param {type:"boolean"}
ALBUM_NAME = "" #@param {type:"string"}
UPLOAD_THREADS = 3 #@param {type:"slider", min:1, max:8, step:1}
UPLOAD_SAVER = False #@param {type:"boolean"}
PAIR_LIVE_PHOTOS = False #@param {type:"boolean"}
DELETE_LOCAL_AFTER_UPLOAD = False #@param {type:"boolean"}
UPLOAD_PROXY = "" #@param {type:"string"}

import html
import json
import os
import subprocess
from IPython.display import display

if not GOTOHP_BIN.exists():
    raise RuntimeError("Vui lòng chạy Bước 1️⃣ trước!")
if not os.path.exists(UPLOAD_SOURCE_PATH):
    display(render_card("❌ Không tìm thấy đường dẫn", f"Đường dẫn <code>{html.escape(UPLOAD_SOURCE_PATH)}</code> không tồn tại.", accent="#ef4444"))
    raise FileNotFoundError(UPLOAD_SOURCE_PATH)

cmd = [
    str(GOTOHP_BIN),
    "upload",
    UPLOAD_SOURCE_PATH,
    "--no-tui",
    "--threads",
    str(int(UPLOAD_THREADS)),
]
if UPLOAD_ACCOUNT.strip():
    cmd += ["--account", UPLOAD_ACCOUNT.strip()]
if RECURSIVE:
    cmd += ["--recursive"]
if ALBUM_NAME.strip():
    cmd += ["--album", ALBUM_NAME.strip()]
if UPLOAD_SAVER:
    cmd += ["--saver"]
if PAIR_LIVE_PHOTOS:
    cmd += ["--pair-live-photos"]
if DELETE_LOCAL_AFTER_UPLOAD:
    cmd += ["--delete"]
if UPLOAD_PROXY.strip():
    cmd += ["--proxy", UPLOAD_PROXY.strip()]

box = display(render_card("⬆️ Đang upload lên Google Photos (0 quota)...", f"Nguồn: <code>{html.escape(UPLOAD_SOURCE_PATH)}</code>"), display_id=True)
proc = subprocess.run(cmd, capture_output=True, text=True)

if not proc.stdout.strip():
    box.update(render_card("❌ Upload thất bại", f"<pre>{html.escape(proc.stderr or 'No output')}</pre>", accent="#ef4444"))
    raise RuntimeError(proc.stderr)

res = json.loads(proc.stdout)
cards = render_kpi_grid(
    [
        ("Tổng mục", str(res.get("total", 0)), "Đã quét", "#818cf8"),
        ("Thành công", str(res.get("succeeded", 0)), "0 quota", "#10b981"),
        ("Bỏ qua (trùng)", str(res.get("skipped", 0)), "Đã có trên GP", "#64748b"),
        ("Thất bại", str(res.get("failed", 0)), "Xem chi tiết JSON", "#ef4444" if res.get("failed", 0) else "#64748b"),
    ]
)
box.update(render_card("✅ Upload hoàn tất" if res.get("failed", 0) == 0 else "⚠️ Upload hoàn tất có lỗi", cards, accent="#10b981" if res.get("failed", 0) == 0 else "#f59e0b"))

In [ ]:
#@title 6️⃣ Tải Backup về máy & Dọn dẹp Credential khỏi VM Colab { display-mode: "form" }
#@markdown Chọn thao tác sau khi chạy xong (khuyên dùng **Xóa config & credential** trước khi đóng tab Colab).
ACTION = "Tải file ZIP backup + báo cáo về máy" #@param ["Tải file ZIP backup + báo cáo về máy", "Chỉ tải file JSON báo cáo", "Xóa config & credential khỏi VM Colab (khi dùng xong)"]
BACKUP_FOLDER = "" #@param {type:"string"}

import html
import os
import shutil
from pathlib import Path
from IPython.display import display

target_dir = Path(BACKUP_FOLDER.strip() or globals().get("DEFAULT_BACKUP_DIR", "/content/gotohp-reclaim"))

if ACTION == "Tải file ZIP backup + báo cáo về máy":
    from google.colab import files  # type: ignore

    if not target_dir.exists() or not any(target_dir.iterdir()):
        display(render_card("ℹ️ Thư mục backup đang trống", f"Không có file nào trong <code>{html.escape(str(target_dir))}</code>.", accent="#f59e0b"))
    else:
        zip_base = "/content/gotohp_reclaim_backups"
        zip_path = shutil.make_archive(zip_base, "zip", str(target_dir))
        display(render_card("📦 Đã nén backup thành công", f"Đang bắt đầu tải <code>{html.escape(zip_path)}</code> ({format_bytes(os.path.getsize(zip_path))})...", accent="#10b981"))
        files.download(zip_path)

elif ACTION == "Chỉ tải file JSON báo cáo":
    from google.colab import files  # type: ignore

    downloaded = []
    for p in ["/content/gotohp_scan_report.json", "/content/gotohp_reclaim_result.json"]:
        if os.path.exists(p):
            files.download(p)
            downloaded.append(p)
    if downloaded:
        display(render_card("📄 Đã tải báo cáo JSON", "<br>".join(f"• <code>{html.escape(p)}</code>" for p in downloaded), accent="#10b981"))
    else:
        display(render_card("ℹ️ Chưa có file báo cáo", "Hãy chạy Bước 3 hoặc Bước 4 trước.", accent="#f59e0b"))

elif ACTION == "Xóa config & credential khỏi VM Colab (khi dùng xong)":
    removed = []
    for p in [
        Path.home() / ".config" / "gotohp",
        Path("/content/gotohp.config"),
        Path("/content/gotohp/gotohp.config"),
    ]:
        if p.exists():
            if p.is_dir():
                shutil.rmtree(p)
            else:
                p.unlink()
            removed.append(str(p))
    if removed:
        display(render_card("🧹 Đã xóa sạch credential khỏi VM Colab", "<br>".join(f"• <code>{html.escape(p)}</code>" for p in removed), accent="#10b981"))
    else:
        display(render_card("✅ Không còn file credential nào trên VM Colab", "Phiên Colab của bạn đã sạch.", accent="#10b981"))